double surface occlusion + 2^20 points sampling refinement

In [1]:

SHARED_ROOT     = "/content/drive/MyDrive/Point Cloud Completion"
ORTAK           = f"{SHARED_ROOT}/Ortak"                  # the common folder
SHARED_DATA_DIR = f"{ORTAK}/data/shapenetcore"           # the ShapeNet zips live here


RUN_TAG         = "s2"
OUT_DIR         = f"{ORTAK}/data_{RUN_TAG}"        # outputs land here


WORK_DIR = "/content/work"
RAW_DIR  = f"{WORK_DIR}/raw"                        # models extracted here from the shared zips
PLY_DIR  = f"{WORK_DIR}/colored_pc"                # PLYs written locally first, copied to OUT_DIR at end


CATEGORIES = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}
SYNSETS = list(CATEGORIES.values())


N_PER_CAT = 20          # models per category
N_POINTS  = 1_048_576   # points CloudCompare samples per model
SEED      = 42          # fixes WHICH models get picked -> same selection for both teammates

HF_REPO_ID = "ShapeNet/ShapeNetCore"   # only used by the OPTIONAL fallback download cell

# --- EPFL clean-mesh path (OPTIONAL second pipeline; speckle path above stays as-is) ---
# The EPFL method removes interior duplicate faces BEFORE sampling, killing the
# salt-and-pepper color speckle. It needs pymeshlab (installed in Cell B).
EPFL_DIR        = f"{WORK_DIR}/colored_pc_epfl"   # clean-mesh PLYs land here (local)
EPFL_OUT_DIR    = f"{ORTAK}/colored_pc_epfl_{RUN_TAG}"  # shared copy of the EPFL outputs
OCCLUSION_THRESHOLD = 0.0   # faces with occlusion < this are also dropped (0 = keep all visible)


In [2]:
# ============================================================
# Cell A — Configuration  (RUN ME FIRST)
# ------------------------------------------------------------
from google.colab import drive
import os

drive.mount('/content/drive')                      # mounts each user's My Drive (+ their shortcuts)

# --- Setup local dirs, and fail LOUDLY if the shared shortcut isn't in place ---
for d in (WORK_DIR, RAW_DIR, PLY_DIR, EPFL_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.isdir(ORTAK), (
    f"\n[STOP] Can't see the shared folder:\n  {ORTAK}\n"
    "Add the 'Point Cloud Completion' shortcut to your My Drive:\n"
    "Drive (web) -> Shared with me -> right-click -> Organize -> Add shortcut to Drive -> My Drive."
)
os.makedirs(OUT_DIR, exist_ok=True)
print("OK. Reading data from:", SHARED_DATA_DIR)
print("    Writing outputs to:", OUT_DIR)


Mounted at /content/drive
OK. Reading data from: /content/drive/MyDrive/Point Cloud Completion/Ortak/data/shapenetcore
    Writing outputs to: /content/drive/MyDrive/Point Cloud Completion/Ortak/data_s2


In [3]:
# ============================================================
# Cell B — Install CloudCompare + a virtual display, and Python deps
# ------------------------------------------------------------
# CloudCompare is a DESKTOP application (not a pip package): we install the binary
# onto the Colab VM with apt. It needs a display even in command-line mode, so we
# also install xvfb (X Virtual FrameBuffer) and will launch it via `xvfb-run`.
# This is the most fragile cell; the REAL functional test is Cell F (sampling one mesh).
# ============================================================
import shutil, subprocess

# System packages (Colab runs Ubuntu 22.04, whose 'universe' repo ships cloudcompare).
subprocess.run("apt-get update -qq", shell=True, check=True)
subprocess.run("apt-get install -y -qq cloudcompare xvfb", shell=True, check=True)

# Python deps not preinstalled on Colab (open3d reads/checks PLY colors; pymeshlab
# runs the EPFL ambient-occlusion step used by the optional clean-mesh path).
subprocess.run("pip install -q -U huggingface_hub open3d pymeshlab", shell=True, check=True)

# Locate the binary (apt installs it as 'CloudCompare', capital C).
CC_BIN = shutil.which("CloudCompare") or shutil.which("cloudcompare")
assert CC_BIN, ("CloudCompare not found after apt install. If this Colab image lacks the "
                "package, we switch to a flatpak/AppImage install in this cell.")
print("CloudCompare:", CC_BIN, "| xvfb-run present:", bool(shutil.which("xvfb-run")))
print("Deps ready. Functional test is Cell F (sampling one mesh).")


CloudCompare: /usr/bin/CloudCompare | xvfb-run present: True
Deps ready. Functional test is Cell F (sampling one mesh).


In [4]:
# ============================================================
# Cell C — Use the shared data in Ortak (NO download)
# ------------------------------------------------------------
import os

missing = [s for s in SYNSETS if not os.path.exists(f"{SHARED_DATA_DIR}/{s}.zip")]
assert not missing, (
    f"\n[STOP] These zips are missing from {SHARED_DATA_DIR}:\n  "
    + ", ".join(f"{s}.zip" for s in missing)
    + "\nCheck the shared folder and your Drive shortcut, or run the OPTIONAL "
      "HF-download cell below to fetch them."
)
for s in SYNSETS:
    p = f"{SHARED_DATA_DIR}/{s}.zip"
    print(f"  {s}.zip: OK ({os.path.getsize(p) / 1e6:.0f} MB)")
print("All 3 category zips present in the shared folder — no download needed.")


  02691156.zip: OK (3359 MB)
  02958343.zip: OK (5685 MB)
  03001627.zip: OK (1965 MB)
All 3 category zips present in the shared folder — no download needed.


In [5]:
# ============================================================
# Cell C-ALT — OPTIONAL fallback: download the 3 zips from HuggingFace
# ------------------------------------------------------------
# DO NOT run this normally. It exists only to (re)populate Ortak/data/shapenetcore/

"""
import os
from huggingface_hub import snapshot_download
from google.colab import userdata

os.makedirs(SHARED_DATA_DIR, exist_ok=True)
snapshot_download(
    repo_id=HF_REPO_ID, repo_type="dataset",
    local_dir=SHARED_DATA_DIR,
    allow_patterns=[f"{s}.zip" for s in SYNSETS],     # only the 3 categories we use
    token=userdata.get("HF_TOKEN"),
)
print("Downloaded the 3 category zips into:", SHARED_DATA_DIR)

"""

'\nimport os\nfrom huggingface_hub import snapshot_download\nfrom google.colab import userdata\n\nos.makedirs(SHARED_DATA_DIR, exist_ok=True)\nsnapshot_download(\n    repo_id=HF_REPO_ID, repo_type="dataset",\n    local_dir=SHARED_DATA_DIR,\n    allow_patterns=[f"{s}.zip" for s in SYNSETS],     # only the 3 categories we use\n    token=userdata.get("HF_TOKEN"),\n)\nprint("Downloaded the 3 category zips into:", SHARED_DATA_DIR)\n\n'

In [6]:
# ============================================================
# Cell D — Inspect the on-disk layout BEFORE writing sampling code
# ------------------------------------------------------------
# We assume nothing about v1 vs v2. Peek inside one zip, extract ONE model, print
# its folder tree, and show the .mtl so we can SEE how textures are referenced
# (the OBJ -> MTL -> image "color chain"). If color goes missing later, it is
# because a link in this chain broke.
# ============================================================
import os, zipfile

probe_synset = SYNSETS[0]
probe_zip = f"{SHARED_DATA_DIR}/{probe_synset}.zip"      # read directly from the shared folder

with zipfile.ZipFile(probe_zip) as zf:
    names = zf.namelist()
    print(f"{probe_synset}.zip — {len(names)} entries. First 15:")
    for n in names[:15]:
        print("   ", n)

    obj_entries = [n for n in names if n.endswith("models/model_normalized.obj")]
    print(f"\nmodel_normalized.obj entries: {len(obj_entries)}")
    one_obj = obj_entries[0]
    model_dir = one_obj[: one_obj.index("models/")]      # "<synset>/<model_id>/"
    members = [n for n in names if n.startswith(model_dir)]
    zf.extractall(path=f"{WORK_DIR}/_probe", members=members)

probe_root = os.path.join(f"{WORK_DIR}/_probe", model_dir)
print(f"\nExtracted one model to: {probe_root}\nTree:")
for root, dirs, files in os.walk(probe_root):
    lvl = root.replace(probe_root, "").count(os.sep)
    print("   " * lvl, os.path.basename(root.rstrip("/")) + "/")
    for f in files:
        print("   " * (lvl + 1), f)

mtl = os.path.join(probe_root, "models", "model_normalized.mtl")
print("\n--- model_normalized.mtl (first 25 lines) ---")
if os.path.exists(mtl):
    with open(mtl) as fh:
        for i, line in zip(range(25), fh):
            print("   ", line.rstrip())
else:
    print("   (no .mtl at expected path — check the tree above)")


02691156.zip — 59867 entries. First 15:
    02691156/
    02691156/10155655850468db78d106ce0a280f87/
    02691156/10155655850468db78d106ce0a280f87/images/
    02691156/10155655850468db78d106ce0a280f87/images/texture0.jpg
    02691156/10155655850468db78d106ce0a280f87/images/texture0.png
    02691156/10155655850468db78d106ce0a280f87/images/texture1.jpg
    02691156/10155655850468db78d106ce0a280f87/images/texture1.png
    02691156/10155655850468db78d106ce0a280f87/images/texture2.jpg
    02691156/10155655850468db78d106ce0a280f87/images/texture2.png
    02691156/10155655850468db78d106ce0a280f87/images/texture3.jpg
    02691156/10155655850468db78d106ce0a280f87/images/texture3.png
    02691156/10155655850468db78d106ce0a280f87/images/texture4.jpg
    02691156/10155655850468db78d106ce0a280f87/images/texture4.png
    02691156/10155655850468db78d106ce0a280f87/images/texture5.jpg
    02691156/10155655850468db78d106ce0a280f87/images/texture5.png

model_normalized.obj entries: 4045

Extracted one mo

In [7]:
# ============================================================
# Cell E — Extract N_PER_CAT models per category (DETERMINISTICALLY)
# ------------------------------------------------------------
# Each model's WHOLE folder (models/ + images/) is extracted intact, so the OBJ's
# relative texture paths resolve and CloudCompare can read color (see Cell D).
# ============================================================
import os, zipfile, random

selected = {}    # synset -> list of (model_id, model_dir_abspath)

for synset in SYNSETS:
    with zipfile.ZipFile(f"{SHARED_DATA_DIR}/{synset}.zip") as zf:   # read from shared folder
        names = zf.namelist()
        # model_id -> its model directory prefix ("<synset>/<model_id>/")
        dir_by_id = {}
        for n in names:
            if n.endswith("models/model_normalized.obj"):
                d = n[: n.index("models/")]
                dir_by_id[d.rstrip("/").split("/")[-1]] = d

        # Reproducible RANDOM selection: sorted() fixes the candidate order across
        # machines, random.Random(SEED) fixes which N_PER_CAT we draw from it.
        rng_sel = random.Random(SEED)
        chosen = rng_sel.sample(sorted(dir_by_id), min(N_PER_CAT, len(dir_by_id)))
        print(f"{synset}: {len(dir_by_id)} models -> extracting {len(chosen)}")

        roots = []
        for mid in chosen:
            d = dir_by_id[mid]
            zf.extractall(path=RAW_DIR, members=[n for n in names if n.startswith(d)])
            roots.append((mid, os.path.join(RAW_DIR, d)))
        selected[synset] = roots

print("\nExtracted per category:", {s: len(v) for s, v in selected.items()})


02691156: 4045 models -> extracting 20
02958343: 3514 models -> extracting 20
03001627: 6778 models -> extracting 20

Extracted per category: {'02691156': 20, '02958343': 20, '03001627': 20}


In [8]:
# ============================================================
# Cell F — The CloudCompare sampling step (+ functional test on ONE model)
# ------------------------------------------------------------
# We keep EPFL's exact command and run it headless via xvfb. Flags:
#   -SILENT              no GUI / no prompts
#   -AUTO_SAVE OFF       don't save after every command
#   -C_EXPORT_FMT PLY    export clouds as PLY (stores per-point r,g,b)
#   -O <obj>             open the mesh
#   -SAMPLE_MESH POINTS N    scatter N color+geometry points over the surface
#   -SAVE_CLOUDS         write the result
# CloudCompare saves "<obj>_SAMPLED_POINTS.ply" NEXT TO the input; we move/rename it.
# ============================================================
import os, glob, shutil, subprocess
import numpy as np, open3d as o3d

def sample_colored_cloud(obj_path, out_ply, n_points=N_POINTS, cc_bin=CC_BIN):
    """Naive colored sampling at N_POINTS (raw, dense; no dual-face fix -> speckle expected)."""
    cmd = (f'xvfb-run -a {cc_bin} -SILENT -AUTO_SAVE OFF -C_EXPORT_FMT PLY '
           f'-O "{obj_path}" -SAMPLE_MESH POINTS {n_points} -SAVE_CLOUDS')
    subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
    produced = [p for p in glob.glob(os.path.join(os.path.dirname(obj_path), "*.ply"))
                if "SAMPLED_POINTS" in os.path.basename(p)]
    if not produced:
        raise RuntimeError(f"No sampled PLY produced for {obj_path}")
    os.makedirs(os.path.dirname(out_ply), exist_ok=True)
    shutil.move(produced[0], out_ply)
    return out_ply

# --- functional smoke test on one model (this exercises the Cell B install end-to-end) ---
ts = SYNSETS[0]
tid, troot = selected[ts][0]
tobj = os.path.join(troot, "models", "model_normalized.obj")
tout = sample_colored_cloud(tobj, f"{PLY_DIR}/{ts}/{tid}.ply")

pc = o3d.io.read_point_cloud(tout)
cols = np.asarray(pc.colors)
print(f"OK -> {tout}  ({os.path.getsize(tout) / 1e6:.1f} MB)")
print(f"points={len(pc.points)}  has_colors={pc.has_colors()}  color_std={cols.std():.4f}")
print("Color present." if pc.has_colors() and cols.std() > 0.01
      else "WARNING: near-constant/no color (textureless model or broken texture chain).")


OK -> /content/work/colored_pc/02691156/ab9e9045e6c7bc6537678474be485ca.ply  (28.3 MB)
points=1048521  has_colors=True  color_std=0.3233
Color present.


In [9]:
# ============================================================
# Cell F-EPFL — Optional EPFL path: remove interior duplicate faces, THEN sample
# ------------------------------------------------------------
# This is the EPFL/MMSPG method (arXiv:2201.06935), ported and kept as a SEPARATE
# path so you can compare against the speckle output from Cell F/G.
#   1) pymeshlab ambient_occlusion -> per-face "how visible from outside" quality
#   2) among faces sharing the SAME 3 vertices, keep only the highest-occlusion one
#      (the exterior-facing face); drop the rest -> kills dual-face color speckle
#   3) rewrite a clean .obj (+ copy its .mtl and textures so color survives)
#   4) sample the CLEAN mesh with CloudCompare -> colored PLY
# NOTE: we intentionally DROP EPFL's final voxelization step (their step 5). Your
# pipeline keeps raw float points; voxelizing would defeat that. Output is a raw
# colored PLY, directly comparable to the speckle PLYs.
# ============================================================
import os, glob, shutil, subprocess
import numpy as np, open3d as o3d

# pymeshlab's ambient_occlusion is memory/state-heavy, so EPFL runs it in a
# separate process. We replicate that: a tiny helper script invoked via subprocess.
_AO_HELPER = "/content/_epfl_ao.py"
with open(_AO_HELPER, "w") as fh:
    fh.write(
        "import sys, numpy as np, pymeshlab\n"
        "in_dir, in_file, fm_file, fq_file = sys.argv[1:5]\n"
        "import os; os.chdir(in_dir)\n"
        "ms = pymeshlab.MeshSet(); ms.load_new_mesh(in_file)\n"
        "ms.compute_scalar_ambient_occlusion()\n"   # pymeshlab >=2022 API (was ambient_occlusion)
        "m = ms.mesh(0)\n"
        "np.save(fm_file, m.face_matrix())\n"
        "np.save(fq_file, m.face_scalar_array())\n"   # pymeshlab >=2022 (was face_quality_array)
    )

def _duplicate_faces_to_delete(face_matrix, face_quality, threshold=0.0):
    """EPFL logic, VECTORIZED for speed (was O(faces^2) -> now O(faces log faces),
    ~400x faster on large meshes). Faces sharing the SAME 3 vertices (any permutation)
    get the same sorted-vertex key; within each group we keep the highest-occlusion
    face and drop the rest. Also drop faces with occlusion < threshold.
    NOTE: on rare DEGENERATE faces (repeated vertices) with TIED occlusion values this
    can pick a different tied face than the original loop -- a few faces out of millions,
    visually irrelevant. Flagged to revisit later if exact parity is ever needed."""
    keys = np.sort(face_matrix, axis=1)                       # canonical per-face key
    order = np.lexsort((keys[:, 2], keys[:, 1], keys[:, 0]))  # group identical keys together
    ks = keys[order]
    change = np.any(ks[1:] != ks[:-1], axis=1)
    grp_start = np.concatenate(([0], np.nonzero(change)[0] + 1))
    grp_end = np.concatenate((grp_start[1:], [len(order)]))

    to_delete = np.zeros(face_matrix.shape[0], dtype=bool)
    for s, e in zip(grp_start, grp_end):
        members = order[s:e]
        if len(members) == 1:
            continue
        members = np.sort(members)                            # original index order
        q = face_quality[members]
        keep = members[np.argmax(q)]                          # first face attaining group max
        to_delete[members[members != keep]] = True
    to_delete = to_delete | (face_quality < threshold)
    return to_delete

def _ensure_face_correspondence(obj_lines, face_matrix):
    """pymeshlab may silently skip some faces on load; align the raw .obj face list
    with pymeshlab's face_matrix so deletion indices line up. Returns None if
    they can't be reconciled (then we skip this model)."""
    nonempty = [(i, x) for i, x in enumerate(obj_lines) if len(x) > 0]
    fobj = [[x.split()[1].split('/')[0], x.split()[2].split('/')[0],
             x.split()[3].split('/')[0]] for _, x in nonempty if x[0] == 'f']
    fobj = np.asarray(fobj, dtype=int) - 1
    if np.array_equal(face_matrix, fobj):
        return obj_lines
    fidx = [i for i, x in nonempty if x[0] == 'f']
    dropped = []
    while fobj.shape[0] > face_matrix.shape[0]:
        corr = (face_matrix == fobj[:face_matrix.shape[0], :]).all(axis=1)
        if not corr.all():
            d = np.nonzero(np.logical_not(corr))[0][0]
            fobj = np.delete(fobj, d, axis=0)
            dropped.append(fidx.pop(d))
        else:
            dropped += fidx[face_matrix.shape[0]:]
            fobj = fobj[:face_matrix.shape[0], :]
    if not np.array_equal(face_matrix, fobj):
        return None
    return list(np.delete(np.asarray(obj_lines), dropped))

def sample_colored_cloud_epfl(obj_path, out_ply, n, n_points=N_POINTS,
                              occlusion_threshold=OCCLUSION_THRESHOLD, cc_bin=CC_BIN):
    """EPFL clean-mesh colored sampling: dedup interior faces, then CloudCompare.
    Returns out_ply. Raises on unrecoverable mesh mismatch (caught by the batch)."""
    in_dir, in_file = os.path.split(obj_path)
    fm_file, fq_file = "face_matrix.npy", "face_quality.npy"

    # 1) ambient occlusion in a separate process (EPFL does this for stability)
    subprocess.run(["python", _AO_HELPER, in_dir, in_file, fm_file, fq_file],
                   capture_output=True, timeout=600)
    fm_path, fq_path = os.path.join(in_dir, fm_file), os.path.join(in_dir, fq_file)
    if not (os.path.exists(fm_path) and os.path.exists(fq_path)):
        raise RuntimeError("ambient_occlusion produced no output (pymeshlab failed).")
    face_matrix = np.load(fm_path); face_quality = np.load(fq_path)
    os.remove(fm_path); os.remove(fq_path)

    # 2) which duplicate/interior faces to drop
    drop = _duplicate_faces_to_delete(face_matrix, face_quality, occlusion_threshold)

    # 3) rewrite a clean .obj with those faces removed
    with open(obj_path, "r") as f:
        obj_lines = f.read().splitlines()
    obj_lines = _ensure_face_correspondence(obj_lines, face_matrix)
    if obj_lines is None:
        raise RuntimeError("face mismatch between pymeshlab and .obj; skipping model.")
    nonempty = [(i, x) for i, x in enumerate(obj_lines) if len(x) > 0]
    fidx = np.asarray([i for i, x in nonempty if x[0] == 'f'])
    obj_lines = list(np.delete(np.asarray(obj_lines), fidx[drop]))

    clean_dir = os.path.join(EPFL_DIR, f"_clean_{n}", "models")
    os.makedirs(clean_dir, exist_ok=True)

    # copy the .mtl referenced in the obj, and point the obj at the local copy
    mtl_i, mtl_line = next((i, x) for i, x in enumerate(obj_lines) if "mtllib" in x)
    mtl_file = next(t for t in mtl_line.split() if t.endswith(".mtl"))
    clean_mtl = f"clean_{n}.mtl"
    shutil.copyfile(os.path.join(in_dir, mtl_file), os.path.join(clean_dir, clean_mtl))
    obj_lines[mtl_i] = f"mtllib {clean_mtl}"

    clean_obj = os.path.join(clean_dir, f"clean_{n}.obj")
    with open(clean_obj, "w") as f:
        f.write("\n".join(obj_lines))

    # copy every texture the .mtl references (so CloudCompare can read color)
    with open(os.path.join(clean_dir, clean_mtl)) as f:
        for line in f.read().splitlines():
            if "map_Kd" in line:
                tex = line.split()[1]
                rel, name = os.path.split(tex)
                dst_dir = os.path.join(clean_dir, rel); os.makedirs(dst_dir, exist_ok=True)
                shutil.copyfile(os.path.join(in_dir, tex), os.path.join(dst_dir, name))

    # 4) sample the CLEAN mesh with CloudCompare (same command as the speckle path)
    cmd = (f'xvfb-run -a {cc_bin} -SILENT -AUTO_SAVE OFF -C_EXPORT_FMT PLY '
           f'-O "{clean_obj}" -SAMPLE_MESH POINTS {n_points} -SAVE_CLOUDS')
    subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
    produced = [p for p in glob.glob(os.path.join(clean_dir, "*.ply"))
                if "SAMPLED_POINTS" in os.path.basename(p)]
    if not produced:
        raise RuntimeError("CloudCompare produced no PLY from the clean mesh.")
    os.makedirs(os.path.dirname(out_ply), exist_ok=True)
    shutil.move(produced[0], out_ply)
    shutil.rmtree(os.path.join(EPFL_DIR, f"_clean_{n}"), ignore_errors=True)  # tidy up
    return out_ply


In [10]:
# --- EPFL smoke test on the SAME single model used in Cell F (ts / tid / troot) ---
# Compare this clean output against the speckle output (tout) from Cell F.
_epfl_out = sample_colored_cloud_epfl(tobj, f"{EPFL_DIR}/{ts}/{tid}.ply", n=0)
_pc = o3d.io.read_point_cloud(_epfl_out)
_cols = np.asarray(_pc.colors)
print(f"EPFL clean -> {_epfl_out}  ({os.path.getsize(_epfl_out)/1e6:.1f} MB)")
print(f"points={len(_pc.points)}  has_colors={_pc.has_colors()}  color_std={_cols.std():.4f}")
print("Compare with the speckle PLY from Cell F to see the dual-face fix.")


EPFL clean -> /content/work/colored_pc_epfl/02691156/ab9e9045e6c7bc6537678474be485ca.ply  (28.3 MB)
points=1048561  has_colors=True  color_std=0.3435
Compare with the speckle PLY from Cell F to see the dual-face fix.


In [11]:
# ============================================================
# Cell G — Sample ALL selected models, robustly (RAW dense clouds only)
# ------------------------------------------------------------
# Per-model try/except so one bad mesh can't abort the batch; failures -> errors list.
# We write ONLY the raw 65,536-point colored PLY straight from CloudCompare -- NO FPS,
# NO normalization (those are deferred to a later derivation step). We record a manifest
# row per success, incl. color_ok (the silent-failure guard: a valid-but-colorless PLY
# means the texture chain broke or the model is textureless).
# ============================================================
import os, traceback
import numpy as np, open3d as o3d
from tqdm import tqdm

NAME_BY_SYNSET = {v: k for k, v in CATEGORIES.items()}

def is_color_ok(ply_path, std_thresh=0.01):
    pc = o3d.io.read_point_cloud(ply_path)
    return bool(pc.has_colors() and np.asarray(pc.colors).std() > std_thresh)

manifest_rows, errors = [], []
epfl_errors = []
epfl_idx = 0   # unique id per clean-mesh temp dir
jobs = [(s, mid, root) for s in SYNSETS for (mid, root) in selected[s]]

for synset, mid, root in tqdm(jobs, desc=f"Sampling {len(jobs)} models"):
    obj = os.path.join(root, "models", "model_normalized.obj")
    out = f"{PLY_DIR}/{synset}/{mid}.ply"
    epfl_out = f"{EPFL_DIR}/{synset}/{mid}.ply"
    try:
        sample_colored_cloud(obj, out)
        pc = o3d.io.read_point_cloud(out)
        row = {
            "synset": synset, "category": NAME_BY_SYNSET[synset], "model_id": mid,
            "source_obj": obj, "output_ply": out,
            "n_points": N_POINTS,
            "actual_points": len(pc.points),
            "color_ok": is_color_ok(out),
        }
        # EPFL clean-mesh path (separate output; failures here don't lose the speckle PLY)
        try:
            sample_colored_cloud_epfl(obj, epfl_out, n=epfl_idx)
            epfl_idx += 1
            row["epfl_ply"] = epfl_out
            row["epfl_color_ok"] = is_color_ok(epfl_out)
        except Exception:
            row["epfl_ply"] = ""
            row["epfl_color_ok"] = False
            epfl_errors.append((synset, mid, traceback.format_exc()))
        manifest_rows.append(row)
    except Exception:
        errors.append((synset, mid, traceback.format_exc()))

n_ok = sum(r["color_ok"] for r in manifest_rows)
print(f"\nSampled {len(manifest_rows)}/{len(jobs)} models; {len(errors)} failed.")
print(f"color_ok (speckle): {n_ok}/{len(manifest_rows)} "
      f"(colorless ones are likely textureless models, not bugs).")
n_epfl = sum(r.get("epfl_color_ok", False) for r in manifest_rows)
print(f"EPFL clean produced: {n_epfl}/{len(manifest_rows)}  "
      f"({len(epfl_errors)} EPFL-step failures -> speckle PLY still kept).")


Sampling 60 models: 100%|██████████| 60/60 [05:37<00:00,  5.62s/it]


Sampled 60/60 models; 0 failed.
color_ok (speckle): 58/60 (colorless ones are likely textureless models, not bugs).
EPFL clean produced: 58/60  (0 EPFL-step failures -> speckle PLY still kept).


In [12]:
# ============================================================
# Cell H — Validate: see colors AND the dual-face speckle (Plotly, inline)
# ------------------------------------------------------------
# color_ok already flagged silent failures across all 60. Here we eyeball a few:
# Open3D's viewer can't run in Colab (no display), so we use a Plotly 3D scatter,
# downsampled for browser speed. Per ADR-0001, visible salt-and-pepper speckle is
# SUCCESS today (it is the problem we fix in a later iteration), not a failure.
# ============================================================
import numpy as np, open3d as o3d
import plotly.graph_objects as go

def show_cloud(ply_path, title, max_pts=25000):
    pc = o3d.io.read_point_cloud(ply_path)
    pts, cols = np.asarray(pc.points), np.asarray(pc.colors)
    if len(pts) > max_pts:                       # keep the browser smooth
        idx = np.random.choice(len(pts), max_pts, replace=False)
        pts, cols = pts[idx], (cols[idx] if len(cols) else cols)
    marker = dict(size=1.5)
    if len(cols):
        marker["color"] = [f"rgb({int(r*255)},{int(g*255)},{int(b*255)})" for r, g, b in cols]
    fig = go.Figure(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                                 mode="markers", marker=marker))
    fig.update_layout(title=title, height=500, scene_aspectmode="data",
                      margin=dict(l=0, r=0, t=30, b=0))
    fig.show()

# one example per category (first successful model)
for synset in SYNSETS:
    row = next((r for r in manifest_rows if r["synset"] == synset), None)
    if row:
        show_cloud(row["output_ply"],
                   f'{row["category"]} — {row["model_id"]} (color_ok={row["color_ok"]})')
    else:
        print(f"No successful model to show for {synset}.")


Output hidden; open in https://colab.research.google.com to view.

In [13]:
# ============================================================
# Cell I — Hand off to the shared Ortak folder
# ------------------------------------------------------------
# Write manifest.csv + errors.log locally, then copy the whole colored_pc/ tree
# (60 small PLYs) from local disk to Ortak/colored_pc/ in ONE batch. This shared
# write IS the transfer: both teammates then see identical files (ADR-0002 / 0003).
# ============================================================
import os, shutil
import pandas as pd

pd.DataFrame(manifest_rows).to_csv(f"{PLY_DIR}/manifest.csv", index=False)
with open(f"{PLY_DIR}/errors.log", "w") as fh:
    if not errors:
        fh.write("No errors.\n")
    for synset, mid, tb in errors:
        fh.write(f"\n=== {synset}/{mid} ===\n{tb}\n")

# Local -> shared Drive (dirs_exist_ok so re-runs overwrite cleanly).
shutil.copytree(PLY_DIR, OUT_DIR, dirs_exist_ok=True)
print("Handoff complete ->", OUT_DIR)
for root, dirs, files in os.walk(OUT_DIR):
    lvl = root.replace(OUT_DIR, "").count(os.sep)
    print("   " * lvl, os.path.basename(root.rstrip("/")) + "/")
    for f in sorted(files)[:6]:
        print("   " * (lvl + 1), f)


Handoff complete -> /content/drive/MyDrive/Point Cloud Completion/Ortak/data_s2
 data_s2/
    errors.log
    manifest.csv
    02691156/
       1580c09f49bb438a7209009cfb89d4bd.ply
       16b2f62791bd9f003554ccf8c30febe7.ply
       172e23ab5b4d189566cf1b4a8fc3914e.ply
       24968851e483feb237678474be485ca.ply
       261093138afff514d8d7812d176664b2.ply
       28da27a6bebc81df62b600da24e0965.ply
    03001627/
       1b80175cc081f3e44e4975e87c20ce53.ply
       1d1c829a54f0ae426cdb122727dd360f.ply
       1de49c5853d04e863c8d0fdfb1cc2535.ply
       2ca91e56bef8cb5034af953b663e921b.ply
       2ef1e7da7f2a124215d65204573ec4.ply
       30f68a6304d6906c9bdca9b7303475c3.ply
    02958343/
       167ec61fc29df46460593c98e3e63028.ply
       17926c1ef484b73e6758a098566bc94e.ply
       18244d93dbd2afbebda733a39f84326d.ply
       263e3ee9f0182cc48e35db9103756ad5.ply
       27d42437168ccd7ddd75f724c0ccbe00.ply
       29b714c4aee36c9d6108f064aff2426d.ply
